# CNN classification on ESC-50

Mel spectrogram (128×128) → `CNNClassifier` (50 classes). Fold 5 = test, fold 4 = validation.

**Prerequisites:** `uv sync`, ESC-50 in `ESC-50-master/`, then `uv run python -m src.data.make_dataset`.

In [ ]:
import torch

from src import config
from src.data.dataloaders import get_dataloaders
from src.models import checkpoints
from src.models.cnn import CNNClassifier
from src.models.trainer import ModelTrainer
from src.visualization.visualize import plot_confusion_matrix, plot_training_history

TRAIN = True  # set False to load best checkpoint and only evaluate

torch.manual_seed(config.RANDOM_SEED)
device = config.get_device(require_cuda=True)
print(config.describe_device(device))
print(f"Manifest: {config.MANIFEST_PATH} (exists={config.MANIFEST_PATH.exists()})")
print("Saved versions:", checkpoints.list_versions("cnn"))

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    "cnn", batch_size=config.BATCH_SIZE_CNN
)
print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

In [ ]:
model = CNNClassifier()
trainer = ModelTrainer(model, device=device, model_name="cnn")

if TRAIN:
    history = trainer.train_classifier(
        train_loader, val_loader, epochs=config.EPOCHS_CNN, save_checkpoints=True
    )
else:
    loaded = checkpoints.load_best("cnn", model)
    if loaded is None:
        raise FileNotFoundError("No checkpoint found. Set TRAIN=True first.")
    history = loaded.get("history", {})

In [ ]:
if history.get("train_loss"):
    plot_training_history(history)

In [ ]:
y_true, y_pred = trainer.predict_classifier(test_loader)
plot_confusion_matrix(y_true, y_pred)